# M3：在 Colab 上训练 ACT（状态输入）

输入 `observation.state`（6 个关节角）+ `observation.environment_state`（方块特征 5 维），不用图像。

**准备**：在 Google Drive 中建目录 `so101_act/`，上传：
- `data/demos.npz`（`export_dataset` 导出的 200 回合）
- `scripts/npz_to_lerobot.py`

训练输出直接写到 Drive（`so101_act/outputs/<run>`），会话断开后可以用最后一个单元续训。
运行时类型选 GPU（L4 / A100 均可）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/so101_act'
%cd $PROJECT
!ls -la

In [ ]:
!pip install -q 'lerobot[dataset,training]==0.6.1'
!python -c "import lerobot, torch; print(lerobot.__version__, torch.__version__, torch.cuda.get_device_name())"

## 1. 转换数据集

两个数据集：全部 200 回合，以及前 50 回合（数据量对比实验）。数据集放在本地盘 `/content/datasets`，转换只需几十秒，不必存到 Drive。

In [ ]:
!python npz_to_lerobot.py demos.npz --repo-id local/so101_pp_200 --root /content/datasets/so101_pp_200
!python npz_to_lerobot.py demos.npz --repo-id local/so101_pp_50 --root /content/datasets/so101_pp_50 --num-episodes 50

## 2. 训练

| 参数 | 值 | 说明 |
|---|---|---|
| `chunk_size` | 50 | 一次预测 1 s 的动作（50 Hz） |
| `n_action_steps` | 50 | 训练时不起作用；推理时可在 `policy_server.py` 中改为时间集成 |
| `batch_size` | 64 | |
| `steps` | 60k | 200 回合约 38 个 epoch；每 10k 步存一个 checkpoint，用闭环成功率挑选 |
| 学习率等 | LeRobot 默认（lr 1e-5，KL 权重 10，dropout 0.1） | |

In [ ]:
STEPS = 60_000
SAVE_FREQ = 10_000

def train(run, dataset):
    cmd = (
        'lerobot-train'
        ' --policy.type=act'
        ' --policy.chunk_size=50'
        ' --policy.n_action_steps=50'
        ' --policy.device=cuda'
        ' --policy.push_to_hub=false'
        f' --dataset.repo_id=local/{dataset}'
        f' --dataset.root=/content/datasets/{dataset}'
        f' --output_dir={PROJECT}/outputs/{run}'
        f' --job_name={run}'
        ' --batch_size=64'
        f' --steps={STEPS}'
        f' --save_freq={SAVE_FREQ}'
        ' --log_freq=500'
        ' --num_workers=4'
        ' --seed=1000'
        ' --wandb.enable=false'
    )
    print(cmd)
    get_ipython().system(cmd)

In [ ]:
train('act_200', 'so101_pp_200')

In [ ]:
train('act_50', 'so101_pp_50')

## 3. 打包 checkpoint

评估只需要每个 checkpoint 的 `pretrained_model/` 目录（不含优化器状态，约 160 MB/个）。下载 zip 后在本机解压到工程的 `outputs/` 下。

In [ ]:
!cd {PROJECT} && zip -qr act_checkpoints.zip outputs/*/checkpoints/*/pretrained_model -x '*/last/*'
!ls -lh {PROJECT}/act_checkpoints.zip

## 断线续训

会话断开后，重新运行前三个单元（挂载、安装、转换），再运行下面的单元（改成要续训的 run）。

In [ ]:
RUN = 'act_200'
!lerobot-train --resume=true --config_path={PROJECT}/outputs/{RUN}/checkpoints/last/pretrained_model/train_config.json